# Hands-on Task: Reliable Tool Caller

Defines one schema, calls the model for five different prompts, and checks that every one comes back valid — retrying if it doesn't.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
import json
from jsonschema import validate, ValidationError
tools = [{"type": "function", "function": {"name": "get_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["city", "unit"]}}}]
schema = tools[0]["function"]["parameters"]
def call_once(prompt, max_retries=3):
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(1, max_retries + 1):
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools, temperature=0)
        calls = response.choices[0].message.tool_calls or []
        if not calls:
            messages.append({"role": "user", "content": "Please call get_weather."})
            continue
        try:
            args = json.loads(calls[0].function.arguments)
            validate(instance=args, schema=schema)
            return args, attempt
        except (json.JSONDecodeError, ValidationError):
            messages.append({"role": "user", "content": "That wasn't valid, please try again."})
    return None, max_retries
prompts = [
    "I'm travelling to Pune tomorrow, what should I pack? Use celsius.",
    "Should I carry an umbrella if I'm heading out in Tokyo today? Fahrenheit please.",
    "Deciding between a t-shirt and a sweater for my walk in Delhi later, in celsius.",
    "I'm hosting an outdoor lunch in London tomorrow, should I move it indoors? I'm using fahrenheit.",
    "Heading out in Mumbai this evening, do I need a jacket? Celsius please.",
]
results = [call_once(p) for p in prompts]
for prompt, (args, attempts) in zip(prompts, results):
    print(prompt, "->", args, f"(attempt {attempts})")
success = sum(1 for args, _ in results if args)
print(f"Success rate: {success}/{len(prompts)}")

## How to Extend
Change one prompt to be even more indirect and see if it still triggers the tool. Lower `max_retries` to 1 and see how many prompts fail on the first try.

## Done When
`success` equals 5 — all five prompts returned valid, schema-matching arguments.